In [ ]:
%pip -q install finlab TA-Lib


In [ ]:
"""Colab cell: API-key authentication and 2330 indicator checks."""
import contextlib, io, os, json, zipfile
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import finlab
from finlab import data
from IPython.display import display

# Key is never written into the notebook, report, or exported files.
_api_key = getpass('FinLab API Key（隱藏輸入）: ').strip()
if not _api_key:
    raise RuntimeError('尚未輸入 API Key')
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        finlab.login(_api_key)
except Exception:
    raise RuntimeError('API Key 登入失敗，請確認 Key 及帳戶權限') from None

FinLab API Key（隱藏輸入）: ··········


1. 加權指數收盤價

In [ ]:
def taiex_close_latest():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數")
    p = p.squeeze("columns").sort_index()

    # 最新收盤指數，單位：點
    taiex_close = float(round(p.iloc[-1], 2))

    return taiex_close

2. 加權指數20日報酬

In [ ]:
def taiex_return_20d():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數")
    p = p.squeeze("columns").sort_index()

    # 近20日報酬，單位：%
    taiex_return_20d_pct = float(round(
        (p.iloc[-1] / p.iloc[-21] - 1) * 100, 2
    ))

    return taiex_return_20d_pct

In [ ]:
taiex_return_20d_pct = taiex_return_20d()
taiex_return_20d_pct

-3.52

3. 加權指數20日均線

In [ ]:
def taiex_ma20_latest():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數")
    p = p.squeeze("columns").sort_index()

    # 最新20日均線，單位：點
    taiex_ma20 = float(round(
        p.rolling(20).mean().iloc[-1], 2
    ))

    return taiex_ma20

In [ ]:
taiex_ma20 = taiex_ma20_latest()
taiex_ma20

48071.61

4. 加權指數60日均線

In [ ]:
def taiex_ma60_latest():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數")
    p = p.squeeze("columns").sort_index()

    # 最新60日均線，單位：點
    taiex_ma60 = float(round(
        p.rolling(60).mean().iloc[-1], 2
    ))

    return taiex_ma60

In [ ]:
taiex_ma60 = taiex_ma60_latest()

5. 大盤20日最大回撤

In [ ]:
def taiex_max_drawdown_20d():

    # 取得加權指數收盤資料
    p = data.get("taiex_total_index:收盤指數")
    p = p.squeeze("columns").sort_index()

    # 最近20個交易日
    recent = p.iloc[-20:]

    # 期間內截至各日的最高指數
    running_high = recent.cummax()

    # 各日相對先前高點的回撤
    drawdown = (recent / running_high - 1) * 100

    # 最大回撤，單位：%；以負值表示跌幅
    taiex_max_drawdown_20d_pct = float(round(
        drawdown.min(), 2
    ))

    return taiex_max_drawdown_20d_pct

6. 外資近20日淨買賣超金額

In [ ]:
def market_foreign_net_buy_20d():

    # 取得上市市場外資淨買賣超金額
    foreign = data.get("institutional_investors_trading_all_market_summary:買賣超")["上市外資及陸資(不含外資自營商)"].sort_index()

    # 最近20日淨買賣超金額，沿用原始金額單位
    foreign_net_buy_20d_amount = float(round(
        foreign.rolling(20).sum().iloc[-1], 2
    ))

    return foreign_net_buy_20d_amount

In [ ]:
foreign_net_buy_20d_amount = market_foreign_net_buy_20d()
print(foreign_net_buy_20d_amount)

-803008363563.0


7.投信近20日淨買賣超金額

In [ ]:
def market_trust_net_buy_20d():

    # 取得上市市場外資淨買賣超金額
    trust = data.get("institutional_investors_trading_all_market_summary:買賣超")["上市投信"].sort_index()

    # 最近20日淨買賣超金額，沿用原始金額單位
    trust_net_buy_20d_amount = float(round(
        trust.rolling(20).sum().iloc[-1], 2
    ))

    return trust_net_buy_20d_amount

In [ ]:
market_trust_net_buy_20d = market_trust_net_buy_20d()
print(market_trust_net_buy_20d)

135997678219.0


8.股票市場漲跌家數比(倍)

In [ ]:
def market_advance_decline_ratio():

    # 取得上市場股票上漲、下跌家數
    up = data.get("limit_up_down_count:上漲")["股票"].sort_index()
    down = data.get("limit_up_down_count:下跌")["股票"].sort_index()

    # 對齊交易日期
    down = down.reindex(up.index)

    # 最新漲跌家數
    up_count = up.iloc[-1]
    down_count = down.iloc[-1]

    if down_count == 0:
        return None

    # 漲跌家數比，單位：倍
    advance_decline_ratio = float(round(up_count / down_count, 2))

    return advance_decline_ratio

In [ ]:
advance_decline_ratio = market_advance_decline_ratio()
advance_decline_ratio

0.33

9. 上市市場融資餘額近5日變化率

In [ ]:
def listed_margin_balance_change_5d():

    # 取得上市市場融資金額餘額
    margin = data.get("margin_balance:融資券總餘額")[
        "上市融資交易金額"
    ].sort_index()

    # 近5個交易日變化率，單位：%
    margin_balance_change_5d_pct = float(round(
        (margin.iloc[-1] / margin.iloc[-6] - 1) * 100, 2
    ))

    return margin_balance_change_5d_pct

In [ ]:
margin_balance_change_5d_pct = listed_margin_balance_change_5d()
print(margin_balance_change_5d_pct)

Daily usage: 880.1 / 5000 MB - margin_balance:融資券總餘額
-1.94


# 新增

## 10. 市場多空排列家數差


使用FinLab台股模式下未套用市場或股票池篩選的 `price:收盤價` 完整價格表，統計最新資料日的多頭與空頭排列標的家數，再回傳兩者差值。此範圍不限上市公司或候選股票池，也未排除ETF等價格表內的其他標的；不是單純的上市公司股票家數。

### 計算方式

- 使用FinLab的 `average(5)`、`average(10)`、`average(30)` 計算三條簡單移動平均。
- `average(n)` 允許窗格內部分缺值，至少需要 `int(n/2)` 筆有效資料；本版5、10、30日均線分別至少需2、5、15筆。這不等同於要求完整n筆資料的 `rolling(n).mean()`。
- 多頭排列：MA5 ≥ MA10 ≥ MA30。
- 空頭排列：MA5 < MA10 < MA30。
- 家數差＝多頭排列家數－空頭排列家數，不對家數差再取均線。
- 僅回傳價格表最新一列的整數結果，單位為「家」；不指定電腦今天日期，也不回傳整段歷史。

### 缺值與相等的處理

- 只要三條均線有效，即可參與排列判斷；不額外要求當日收盤價有效，因此可能計入當天缺價但仍可算出均線的標的。
- 任一均線缺值時，不計入多頭或空頭；最新一天完全沒有標的可計算三條均線時會報錯。
- 三條均線全部相等時，依本版的≥條件會計入多頭；未符合任一完整排列者不計入兩方。

### 數值意義

- 正值：符合多頭排列的標的較多。
- 負值：符合空頭排列的標的較多。
- 0：兩方家數相同，不代表所有標的沒有趨勢。
- 例如-1,097代表空頭排列比多頭排列多1,097家，不是下跌1,097點，也不是百分比。
- 這是均線排列結構，不是當天上漲家數減下跌家數；單筆結果也不能判斷市場正在改善或惡化，須與前期比較。

### 核對紀錄

依本次提供的2026-11-03資料，多頭373家、空頭1,470家，差值-1,097，與FinLab market-pulse當日顯示一致；其中19個計入標的當日缺少收盤價。目前僅確認此日期的總數一致，尚未逐日核對網站後台公式或逐檔名單，不能把全部差異歸因於這19個標的。


In [ ]:
def market_alignment():
    from finlab import data

    # 台股模式下取得未篩選價格表，不限上市或候選股票池，也不排除ETF
    with data.universe():
        close = data.get("price:收盤價").sort_index()

    # 簡單移動平均；average(n)允許部分缺值，至少需int(n/2)筆有效資料
    # 本版5、10、30日均線至少需2、5、15筆，並非要求完整n筆
    ma5 = close.average(5)
    ma10 = close.average(10)
    ma30 = close.average(30)

    # 確認最新資料日至少有標的可計算三條均線，避免資料全缺卻回傳0
    if close.empty or not (
        ma5.notna() & ma10.notna() & ma30.notna()
    ).iloc[-1].any():
        raise ValueError("最新交易日的均線資料不足")

    # 多頭排列：MA5 >= MA10 >= MA30；三線全相等也計入多頭
    bullish = (ma5 >= ma10) & (ma10 >= ma30)

    # 空頭排列：MA5 < MA10 < MA30；未符合任一完整排列者不計入兩方
    bearish = (ma5 < ma10) & (ma10 < ma30)

    # 不額外排除當日缺價標的；只要均線有效且符合排列，仍計入家數
    # 任一均線為NaN時，比較不成立，不會計入多頭或空頭
    bullish_count = int(bullish.iloc[-1].sum())
    bearish_count = int(bearish.iloc[-1].sum())

    # 只回傳最新資料日的整數家數差，不再對差值取均線
    # 正值＝多頭較多；負值＝空頭較多；0＝兩方相同，不代表沒有趨勢
    # 這不是當日漲跌家數差；是否改善或轉弱仍需與前期比較
    alignment_count_diff = bullish_count - bearish_count

    return alignment_count_diff


In [ ]:
alignment = market_alignment()
alignment

-1097

## 11. 生命線：加權報酬指數與月線的關係

生命線的重點是看「大盤是否站在月線上」。依圖片說明，這裡使用**發行量加權股價報酬指數**，不是一般加權價格指數。報酬指數包含股息再投入效果；同一條報酬指數的20個交易日簡單移動平均，包含當天。

### 回傳的兩個數值

| 欄位 | 計算方式與含意 |
|---|---|
| `index_vs_ma20_pct` | （最新報酬指數／最新20日均線－1）×100%；正值表示在月線上方，負值表示在月線下方 |
| `ma20_change_5d_pct` | （最新20日均線／5個交易日前20日均線－1）×100%；正值表示月線較5日前上升，負值表示下降 |

例如`index_vs_ma20_pct=-8.5`，代表最新報酬指數比月線低8.5%，不是當日下跌8.5%，也不是下跌8.5點。若指數為91,500、月線為100,000，計算結果就是-8.5%。圖片中的「跌破月線8.5%」在本版以這個負乖離率表示。

### 搭配判讀

| 報酬指數位置 | 月線近5日方向 | 可描述的狀態 |
|---|---|---|
| 月線上方 | 上升 | 指數高於近期平均，月線也比5日前提高，價格位置與均線方向一致偏強 |
| 月線上方 | 下降 | 指數在月線上方，但月線仍低於5日前，兩項證據尚未一致 |
| 月線下方 | 上升 | 指數低於近期平均，但月線仍高於5日前，可能是回落，不能單憑位置判定持續空頭 |
| 月線下方 | 下降 | 指數低於近期平均，月線也較5日前降低，兩項證據一致偏弱 |

數值等於0時代表對應位置或端點相等，不套用正負分類。單日顯示負值只表示目前在月線下，不代表今天才由上往下穿越月線；本版沒有輸出新交叉事件。

- 生命線看的是上市加權市場的趨勢背景，不代表每一檔股票或上櫃市場都同步轉強／轉弱。

呼叫方式：

```python
lifeline = listed_market_lifeline()
print(lifeline)
```


In [ ]:
def listed_market_lifeline():
    from finlab import data
    import pandas as pd

    # 生命線觀察加權「報酬指數」是否站在月線上，包含股息再投入效果。
    # 與新聞常見的加權價格指數不同，不能混用兩者的指數與均線。
    index = data.get("benchmark_return:發行量加權股價報酬指數")
    index = index.squeeze("columns").sort_index()

    if not isinstance(index, pd.Series):
        raise ValueError("預期加權報酬指數只有一個數值欄位")

    # 本版月線定義為20個交易日的簡單移動平均，包含當天。
    ma20 = index.rolling(20).mean()

    # 20日均線加上5日比較，共需最近25筆完整資料；不跳過缺值。
    if (
        len(index) < 25
        or index.iloc[-25:].isna().any()
        or (index.iloc[-25:] <= 0).any()
    ):
        raise ValueError("計算生命線需要最近25個交易日的有效報酬指數")

    # 大盤距月線幅度，單位：%。例如-8.5代表低於月線8.5%。
    # 這是目前的位置，不代表今天才跌破，亦不是單日跌幅。
    index_vs_ma20_pct = (index.iloc[-1] / ma20.iloc[-1] - 1) * 100

    # 月線近5日變化率，單位：%；正值表示比5個交易日前上升。
    # 只比較兩個端點，不表示期間每天都上升或下降。
    ma20_change_5d_pct = (ma20.iloc[-1] / ma20.iloc[-6] - 1) * 100

    # 僅回傳最新資料日的兩個數值，計算完成後才取小數第二位。
    return {
        "index_vs_ma20_pct": round(float(index_vs_ma20_pct), 2),
        "ma20_change_5d_pct": round(float(ma20_change_5d_pct), 2)
    }


In [ ]:
# 生命線
lifeline = listed_market_lifeline()
print(lifeline)

Daily usage: 269.2 / 5000 MB - benchmark_return:發行量加權股價報酬指數
{'index_vs_ma20_pct': -8.48, 'ma20_change_5d_pct': -1.38}


In [ ]:
lifeline

{'index_vs_ma20_pct': -8.48, 'ma20_change_5d_pct': -1.38}